# Policy RAG Assistant

### Install dependencies

In [ ]:
%%capture
!pip install -U openai langchain langgraph langchain-openai langchain-community faiss-cpu pypdf tiktoken

### Import libraries

In [ ]:
%%capture
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

from langgraph.graph import StateGraph, END
from langchain_core.runnables import RunnableLambda

from typing import TypedDict, List, Optional
import glob

### Add your OpenAI API key.


In [ ]:
import os
os.environ['AZURE_OPENAI_API_KEY'] = ''
os.environ['AZURE_OPENAI_ENDPOINT'] = ''
os.environ['OPENAI_API_VERSION'] = ''

### Initialize the language model and the embedding model.

In [ ]:
llm = AzureChatOpenAI(model="gpt-4o-mini", temperature=0)
embedder = AzureOpenAIEmbeddings(model="text-embedding-3-small")

### Load all policy PDFs.

In [ ]:
pdf_dir = "policies"
pdf_paths = sorted(glob.glob(os.path.join(pdf_dir, "*.pdf")))
assert pdf_paths, f"No PDFs found in {pdf_dir}. Please add your policy PDFs first."

raw_docs: List[Document] = []
for path in pdf_paths:
    loader = PyPDFLoader(path)
    raw_docs.extend(loader.load())  # each Document has metadata: {'source': path, 'page': n}

len(raw_docs), [os.path.basename(p) for p in pdf_paths]


### Chunk the documents and keep source/page metadata for citations.

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = splitter.split_documents(raw_docs)
for d in docs:
    d.metadata = {
        "source": os.path.basename(d.metadata.get("source", "")),
        "page": int(d.metadata.get("page", -1)),
    }

len(docs)

### Build the vector store (FAISS) and a retriever (k=4).

In [ ]:
vectorstore = FAISS.from_documents(docs, embedder)
retriever = vectorstore.as_retriever(search_kwargs={"k":3})


### Define the agent’s “state” and the two actions: retrieve and generate:

In [ ]:
class RAGState(TypedDict):
    question: str
    docs: Optional[List[Document]]
    answer: Optional[str]

def retrieve(state: RAGState) -> RAGState:
    question = state["question"]
    results = retriever.invoke(question)
    return {"question":question, "docs":results, "answer": None}

def _format_context_for_llm(docs: List[Document]) -> str:
    lines = []
    for i, d in enumerate(docs, start=1):
        src = d.metadata.get("source", "unknown.pdf")
        page = d.metadata.get("page", -1)
        lines.append(f"[{i}] {src} (p.{page})\n{d.page_content}")
    return "\n\n".join(lines)

def _format_citations(docs: List[Document]) -> str:
    seen = set()
    items = []
    for i, d in enumerate(docs, start=1):
        key = (d.metadata.get("source"), d.metadata.get("page"))
        if key not in seen:
            seen.add(key)
            items.append(f"[{i}] {key[0]} (p.{key[1]})")
    return "\n".join(items)

def generate(state: RAGState) -> RAGState:
    question = state["question"]
    docs = state["docs"] or []
    context = "\n\n".join(d.page_content for d in docs)

    system = (
        "You are a brokerage assistant. Answer using ONLY the provided context chunks. "
        "If the answer is not in the context, say 'I don't know based on the provided documents.' "
        "Avoid speculation. Be concise. When relevant, include short quotes from the context."
    )
    user = (
        f"Context chunks (each tagged):\n{context}\n\n"
        f"Question: {question}\n\n"
        f"Respond in this format:\n"
        f"Answer: <your concise answer>\n"
        f"Sources:\n- <tag + file + page for each supporting chunk>"
    )

    resp = llm.invoke([
            {"role":"system", "content": system},
            {"role":"user", "content": user}
        ])

    # Fallback: append computed sources if the model forgot.
    fallback_sources = _format_citations(docs)
    if "Sources:" not in resp:
        resp = f"{resp.content}\n\nSources:\n{fallback_sources.content}"
    return {"question": question, "docs": docs, "answer": resp}


### Connect all actions into a runnable graph

In [ ]:
builder = StateGraph(RAGState)
builder.add_node("retrieve", RunnableLambda(retrieve))
builder.add_node("generate", RunnableLambda(generate))
builder.set_entry_point("retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)
# add your code here
graph = builder.compile()


### Test: Ask the agent a question and show answers and sources

In [ ]:
question = "Does any policy cover accidental damage during organised races?"
result = graph.invoke({"question": question, "docs": None, "answer": None})

print("=== ANSWER ===")
print(result["answer"])


Inspect the retrieved evidence (to build trust and debug relevance):

In [ ]:
for i, d in enumerate(result["docs"], start=1):
    src = d.metadata.get("source")
    page = d.metadata.get("page")
    preview = d.page_content[:600].replace("\n", " ")
    print(f"\n[{i}] {src} (p.{page})\n{preview}{'...' if len(d.page_content) > 600 else ''}")
